In [1]:
!pip install datasets transformers accelerate peft trl bitsandbytes python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.0 MB/s eta 0:00:00


In [2]:
# Import packages
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

In [3]:
# Load environment and save HuggingFace token
from dotenv import load_dotenv
load_dotenv("hfkey.txt")
from huggingface_hub import login
token = os.getenv("HF_TOKEN")
login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Model name from Llama
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# Fine-tuned model name
new_model = "llama-fine-tuning-financial-customer-service"

# Load dataset. For this fine tuning, dataset from local
dataset = load_dataset("json",
                       data_files="fcsdataset.jsonl",
                       split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
# reducing memory usage, using QLoRA
bnbconfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

In [6]:
# download Llama model from HuggingFace Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnbconfig,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [7]:
# get processing or tokenizer, prepare the inputs in the model.
processing_class = AutoTokenizer.from_pretrained(model_name)
processing_class.pad_token = processing_class.eos_token

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [8]:
# balancing between performance and computational
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [9]:
# Supervised Fine-Tuning configuration
sft_config = SFTConfig(
    output_dir="./results",  # save output directory
    num_train_epochs=2,  # every 2 epochs training
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    save_steps=10,
    logging_steps=10,
    learning_rate=0.002,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="linear",
    report_to="none",  # disable wandb or tensorboard
)

# Method of training fine-tuned model
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=processing_class,
    args=sft_config,
)

Adding EOS to train dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

In [10]:
# Run training fine-tuned model
trainer.train()

# Save trained model and processing class/tokenizer
trainer.model.save_pretrained(new_model)
trainer.processing_class.save_pretrained(new_model)

print("Fine-tuning are successfull.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,2.398700
20,0.797000
30,0.252700
40,0.068900


Fine-tuning are successfull.


In [11]:
# Create HuggingFace repository
from huggingface_hub import create_repo
create_repo("budionosan/llama-finetuning-financial-customer-service")

RepoUrl('https://huggingface.co/budionosan/llama-finetuning-financial-customer-service', endpoint='https://huggingface.co', repo_type='model', repo_id='budionosan/llama-finetuning-financial-customer-service')

In [12]:
# Upload finetuning mode to HuggingFace repository
from huggingface_hub import upload_folder
upload_folder(
    folder_path="/content/llama-fine-tuning-financial-customer-service/",
    repo_id="budionosan/llama-finetuning-financial-customer-service",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...er-service/tokenizer.json:   2%|1         |  295kB / 17.2MB            

  ...adapter_model.safetensors:   3%|3         |  458kB / 13.6MB            

CommitInfo(commit_url='https://huggingface.co/budionosan/llama-finetuning-financial-customer-service/commit/17c1a936b655a0ea31cb5f83e7ccb7ee0c623c74', commit_message='Upload folder using huggingface_hub', commit_description='', oid='17c1a936b655a0ea31cb5f83e7ccb7ee0c623c74', pr_url=None, repo_url=RepoUrl('https://huggingface.co/budionosan/llama-finetuning-financial-customer-service', endpoint='https://huggingface.co', repo_type='model', repo_id='budionosan/llama-finetuning-financial-customer-service'), pr_revision=None, pr_num=None)